# NLP HW2 — A100, один воспроизводимый эксперимент

Выберите A100 → **Runtime → Run all**, подтвердите Drive. Единственный внешний вход — `colab_training_data.zip` в `MyDrive/nlp_hw2/`. Весь исполняемый код находится в ячейках: GitHub/наши Python-файлы не загружаются. Нет pretrained weights, HF/Comet/API keys, compile или выполнения generated-программ.

**Предел A100 — 6 часов от первой ячейки**, последние30 минут — сохранение/экспорт; для отдельной T4 сессии оставлено30 минут. Время от подключения GPU до Run all также оплачивается, но код его не видит. Конец Python **не отключает runtime**: после итоговой проверки вручную Disconnect and delete runtime. При внезапном уничтожении VM последнюю запись гарантировать нельзя.

Основная модель6+6,d768,12heads,FFN3072,BPE8192 — **106,321,920 параметров**. Облачный критерий выбора — CE; численная accuracy и public pass@1 неизвестны до локальной проверки. См. последний вывод: там будут пути результатов и статус проверки сохранённых экспортов.

## 1. Настройки и начало бюджета

In [ ]:
import os, sys, time

SESSION_STARTED_UNIX = time.time()
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError(
        "Restart session, then Run all: CUDA must be restricted before initialization"
    )
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path
from datetime import datetime, timezone

DRIVE_ROOT = Path("/content/drive/MyDrive/nlp_hw2")
DATA_ARCHIVE = DRIVE_ROOT / "colab_training_data.zip"
RUN_NAME = "large_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RESUME_FROM = None  # Path('.../runs/<old>/checkpoints/last.json'); keep RUN_NAME of that run
A100_BUDGET_SECONDS = 6 * 3600  # cumulative known A100 time across resume, never increase
FINAL_RESERVE_SECONDS = 30 * 60
RUN_PHASES = True
LOCAL_ROOT = Path("/content/nlp_hw2")
NOTEBOOK_REVISION = "colab_a100_20261023_v1"
IMPLEMENTATION_SHA256 = '383582e145019ff09bd58babbbd153320aa441444b204efdbdc1d31d197fbbd4'
DATA_ARCHIVE_SHA256 = "6eed1ad2118aaa0aa7d3c600902101a2c83034d902d9cb21efc9e42fd2933f15"
DATA_MANIFEST_SHA256 = "0ea76d95a9f005f9eb4531cef6b472aa0a4fef9596a7658073853ff311c1cdda"
CONFIG = {
    "seed": 42,
    "model": {
        "activation_dropout": 0.0,
        "activation_function": "gelu",
        "architectures": ["BartForConditionalGeneration"],
        "attention_dropout": 0.0,
        "bos_token_id": 1,
        "classifier_dropout": 0.0,
        "d_model": 768,
        "decoder_attention_heads": 12,
        "decoder_ffn_dim": 3072,
        "decoder_layerdrop": 0.0,
        "decoder_layers": 6,
        "decoder_start_token_id": 1,
        "dropout": 0.1,
        "encoder_attention_heads": 12,
        "encoder_ffn_dim": 3072,
        "encoder_layerdrop": 0.0,
        "encoder_layers": 6,
        "eos_token_id": 2,
        "forced_eos_token_id": None,
        "id2label": {"0": "LABEL_0", "1": "LABEL_1", "2": "LABEL_2"},
        "init_std": 0.02,
        "is_encoder_decoder": True,
        "label2id": {"LABEL_0": 0, "LABEL_1": 1, "LABEL_2": 2},
        "max_position_embeddings": 512,
        "model_type": "bart",
        "num_hidden_layers": 6,
        "pad_token_id": 0,
        "scale_embedding": False,
        "torch_dtype": "float32",
        "transformers_version": "4.46.3",
        "use_cache": True,
        "vocab_size": 8192,
    },
    "training": {
        "effective_batch": 256,
        "physical_batch_candidates": [32, 64, 128, 256],
        "peak_lrs": [0.0002, 0.0005],
        "weight_decay": 0.01,
        "gradient_clip": 1.0,
        "source_limit": 192,
        "target_limit": 384,
        "warmup_fraction": 0.03,
        "cosine_floor": 0.1,
        "ce_every_seconds": 1500,
        "generation_every_seconds": 3600,
        "checkpoint_every_seconds": 1500,
        "max_candidates": 3,
        "ce_batch_size": 32,
    },
    "budget": {
        "a100_seconds": 21600,
        "prepare_seconds": 1200,
        "benchmark_seconds": 600,
        "trials_seconds": 2400,
        "main_seconds": 16200,
        "final_reserve_seconds": 1800,
    },
    "generation": {
        "num_beams": 1,
        "do_sample": False,
        "max_new_tokens": 256,
        "batch_size": 32,
        "length_penalty": 1.0,
        "early_stopping": False,
        "use_cache": True,
        "forced_eos_token_id": None,
        "pad_token_id": 0,
        "bos_token_id": 1,
        "eos_token_id": 2,
        "decoder_start_token_id": 1,
    },
}
print("Timer started:", SESSION_STARTED_UNIX, "A100 budget:", A100_BUDGET_SECONDS, "s")


## 2. Drive и безопасные записи
Данные читаются с локального `/content`; Drive используется для входного ZIP и результатов. `last.json` указывает на полное проверенное состояние; `last.previous.json` — предыдущая исправная копия. Resume сохраняет расписание и вычитает уже записанное время A100 из общего лимита. Время между последней записью и гибелью VM код точно восстановить не может: при resume уменьшите общий бюджет дополнительно на известный вам расход.

In [ ]:
import json, hashlib, shutil, uuid, zipfile, subprocess, signal, traceback, math, gc, platform

"""Verified writes, immutable full checkpoints and wall-clock budget accounting."""
import hashlib
import json
import os
from pathlib import Path
import shutil
import time
import uuid
import zipfile


class DeadlineReached(RuntimeError):
    pass


class Deadline:
    def __init__(self, start, total=21600, reserve=1800, prior=0):
        self.start = start
        self.total = total
        self.prior = prior
        self.hard = start + max(0, total - prior)
        self.cutoff = self.hard - reserve

    def remaining(self, final=False):
        return (self.hard if final else self.cutoff) - time.time()

    def require(self, seconds=0, final=False):
        if self.remaining(final) <= seconds:
            raise DeadlineReached("Wall-clock reserve reached")

    def elapsed(self):
        return self.prior + max(0, time.time() - self.start)


def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while b := f.read(8 * 1024 * 1024):
            h.update(b)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(path)


def publish_file(source, destination):
    """Verify new bytes before replacing a destination. Old bytes survive copy failures."""
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp = destination.with_name(destination.name + ".upload-" + uuid.uuid4().hex)
    try:
        expected = sha(source)
        shutil.copyfile(source, tmp)
        if sha(tmp) != expected:
            raise IOError(f"Drive copy checksum mismatch: {destination}")
        tmp.replace(destination)
        return expected
    finally:
        if tmp.exists():
            tmp.unlink()


def publish_tree(source, destination):
    source, destination = Path(source), Path(destination)
    if destination.exists():
        raise FileExistsError(destination)
    temp = destination.with_name(destination.name + ".upload-" + uuid.uuid4().hex)
    temp.mkdir(parents=True)
    try:
        for p in source.rglob("*"):
            if p.is_file():
                publish_file(p, temp / p.relative_to(source))
        for p in source.rglob("*"):
            if p.is_file() and sha(p) != sha(temp / p.relative_to(source)):
                raise IOError("Export mismatch")
        temp.rename(destination)
    except BaseException:
        # A partial .upload tree is never advertised as a valid export.
        raise


def publish_checkpoint(local, remote_dir, metadata):
    remote_dir = Path(remote_dir)
    remote_dir.mkdir(parents=True, exist_ok=True)
    name = f'last_{metadata["updates"]:09d}_{uuid.uuid4().hex[:8]}.pt'
    digest = publish_file(local, remote_dir / name)
    record = {**metadata, "file": name, "sha256": digest}
    if "session_started_unix" in metadata:
        record["a100_elapsed_seconds"] = metadata.get("prior_elapsed_seconds", 0) + max(
            0, time.time() - metadata["session_started_unix"]
        )
    pointer = remote_dir / "last.json"
    if pointer.exists():
        publish_file(pointer, remote_dir / "last.previous.json")
    temp = Path(local).parent / "last_pointer.json"
    write_json(temp, record)
    publish_file(temp, pointer)
    keep = {name}
    prev = remote_dir / "last.previous.json"
    if prev.exists():
        keep.add(json.loads(prev.read_text())["file"])
    for p in remote_dir.glob("last_*.pt"):
        if p.name not in keep:
            p.unlink()
    return record


def resolve_checkpoint(pointer):
    pointer = Path(pointer)
    r = json.loads(pointer.read_text())
    name = r["file"]
    if Path(name).name != name:
        raise ValueError("Checkpoint pointer must name a local sibling")
    p = pointer.parent / name
    if sha(p) != r["sha256"]:
        raise IOError("Resume checksum mismatch; try last.previous.json")
    return p, r


def verify_tree(directory, manifest_name="manifest.json"):
    directory = Path(directory)
    m = json.loads((directory / manifest_name).read_text())
    for name, record in m["files"].items():
        p = directory / name
        if directory.resolve() not in p.resolve().parents:
            raise ValueError("Unsafe manifest path")
        if p.stat().st_size != record["bytes"] or sha(p) != record["sha256"]:
            raise IOError(f"Asset mismatch: {name}")
    return m


def extract_zip(source, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as z:
        for info in z.infolist():
            target = (destination / info.filename).resolve()
            if destination.resolve() not in target.parents:
                raise ValueError("Unsafe ZIP member")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("ZIP symlinks not allowed")
        z.extractall(destination)


DRIVE_ERROR = None


def drive_failure(exc):
    global DRIVE_ERROR
    DRIVE_ERROR = repr(exc)
    print("DRIVE WRITE FAILED:", DRIVE_ERROR, "Local files remain in", LOCAL_ROOT, flush=True)
    write_json(LOCAL_ROOT / "drive_error.json", {"error": DRIVE_ERROR, "time_unix": time.time()})


if RUN_PHASES:
    from google.colab import drive

    drive.mount("/content/drive")
    if not 0 < A100_BUDGET_SECONDS <= 21600 or FINAL_RESERVE_SECONDS < 1800:
        raise ValueError("A100<=6h and final reserve>=30min are required")
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    PRIOR_ELAPSED = 0.0
    RESUME_LOCAL = None
    RESUME_RECORD = None
    if RESUME_FROM is not None:
        RESUME_SOURCE, RESUME_RECORD = resolve_checkpoint(RESUME_FROM)
        PRIOR_ELAPSED = float(RESUME_RECORD["a100_elapsed_seconds"])
    BUDGET = Deadline(
        SESSION_STARTED_UNIX, A100_BUDGET_SECONDS, FINAL_RESERVE_SECONDS, PRIOR_ELAPSED
    )
    DRIVE_RUN = DRIVE_ROOT / "runs" / RUN_NAME
    if DRIVE_RUN.exists() and RESUME_FROM is None:
        raise FileExistsError("Choose a new RUN_NAME or explicit RESUME_FROM")
    if RESUME_FROM is not None and Path(RESUME_FROM).parent.parent.resolve() != DRIVE_RUN.resolve():
        raise ValueError("Resume must use the same RUN_NAME and its last.json/last.previous.json")
    RUN = LOCAL_ROOT / "runs" / RUN_NAME
    if RUN.exists():
        raise FileExistsError("Local run exists: restart clean runtime before resume")
    for name in ["configs", "logs", "checkpoints", "validation_predictions", "exports"]:
        (RUN / name).mkdir(parents=True)
    DRIVE_RUN.mkdir(parents=True, exist_ok=True)
    # Preserve one absolute deadline across resumed sessions. Counting downtime is
    # conservative: a killed runtime cannot report its exact last GPU-active second.
    budget_path = DRIVE_RUN / "budget.json"
    if RESUME_FROM is not None:
        ledger = json.loads(budget_path.read_text())
        if ledger["total_seconds"] != A100_BUDGET_SECONDS:
            raise ValueError("Resume cannot reset or change the original budget")
        PRIOR_ELAPSED = max(
            PRIOR_ELAPSED, SESSION_STARTED_UNIX - ledger["first_session_started_unix"]
        )
        BUDGET = Deadline(
            SESSION_STARTED_UNIX, A100_BUDGET_SECONDS, FINAL_RESERVE_SECONDS, PRIOR_ELAPSED
        )
    else:
        budget_local = RUN / "configs/budget.json"
        write_json(
            budget_local,
            {
                "first_session_started_unix": SESSION_STARTED_UNIX,
                "total_seconds": A100_BUDGET_SECONDS,
                "absolute_deadline_unix": BUDGET.hard,
                "resume_counts_downtime_conservatively": True,
            },
        )
        publish_file(budget_local, budget_path)
    BUDGET.require(120)
    if RESUME_FROM is not None:
        RESUME_LOCAL = RUN / "checkpoints/resume_input.pt"
        publish_file(RESUME_SOURCE, RESUME_LOCAL)
        if sha(RESUME_LOCAL) != RESUME_RECORD["sha256"]:
            raise IOError("Resume copy mismatch")
    print(
        "Local:", RUN, "Drive:", DRIVE_RUN, "remaining known A100 seconds:", BUDGET.remaining(True)
    )


## 3. Зависимости и проверка A100
CUDA-enabled PyTorch предоставляет Colab: огромный wheel torch не переустанавливается. Проверяются реальное имя GPU и аппаратная BF16-поддержка. Версии сохраняются. Установка, копирование и все проверки входят в бюджет.

In [ ]:
REQUIREMENTS = """# Keep Colab's CUDA-enabled torch. Exact observed versions are saved with every run.
transformers==4.46.3
tokenizers==0.20.3
safetensors>=0.4.3,<1
numpy>=1.26,<3
pyarrow>=18.1,<25
"""
if RUN_PHASES:
    BUDGET.require(180)
    t = time.perf_counter()
    req = LOCAL_ROOT / "requirements-colab.txt"
    req.write_text(REQUIREMENTS)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-q",
            "-r",
            str(req),
        ],
        check=True,
        timeout=max(1, min(600, BUDGET.remaining() - 120)),
    )
    DEPENDENCY_SECONDS = time.perf_counter() - t
    import importlib.metadata
    import torch

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Select one A100 before Run all")
    GPU = torch.cuda.get_device_name(0)
    if (
        "A100" not in GPU
        or torch.cuda.get_device_capability(0)[0] < 8
        or not torch.cuda.is_bf16_supported()
    ):
        raise RuntimeError("Hardware A100/BF16 required; no silent T4 fallback")
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    DEVICE = torch.device("cuda")
    DTYPE = torch.bfloat16
    VERSIONS = {
        n: importlib.metadata.version(n)
        for n in ["torch", "transformers", "tokenizers", "numpy", "pyarrow", "safetensors"]
    }
    print(
        "GPU:",
        GPU,
        "CUDA:",
        torch.version.cuda,
        "Python:",
        platform.python_version(),
        "versions:",
        VERSIONS,
        "install seconds:",
        DEPENDENCY_SECONDS,
    )
    write_json(
        RUN / "logs/environment.json",
        {
            "gpu": GPU,
            "cuda": torch.version.cuda,
            "versions": VERSIONS,
            "dependency_seconds": DEPENDENCY_SECONDS,
            "python": sys.version,
            "notebook_revision": NOTEBOOK_REVISION,
            "implementation_sha256": IMPLEMENTATION_SHA256,
        },
    )
    (RUN / "logs/environment-freeze.txt").write_text(
        subprocess.run(
            [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True
        ).stdout
    )


## 4. Данные: локальный mmap, split, manifests
Архив:5,820,047 разрешённых пар, frozen BPE8192, validation1500 и фиксированные512/493 references. Целиком тексты train в RAM не читаются. Числовые индексы NumPy перемешиваются и сортируются по длине небольшими блоками; неполный последний update сохраняется. Проверяются SHA всех файлов и отсутствие train-групп hash%200=0.

In [ ]:
import random, contextlib, copy
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BartConfig, BartForConditionalGeneration
from tokenizers import Tokenizer

"""Frozen compact token caches, identical bucket order and decoder shift/padding."""

import math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Sampler

PAD, BOS, EOS = 0, 1, 2


class TokenDataset(Dataset):
    def __init__(self, directory, split="train", indices=None):
        directory = Path(directory)
        self.source = np.load(directory / f"{split}_source.npy", mmap_mode="r")
        self.target = np.load(directory / f"{split}_target.npy", mmap_mode="r")
        self.source_offsets = np.load(directory / f"{split}_source_offsets.npy", mmap_mode="r")
        self.target_offsets = np.load(directory / f"{split}_target_offsets.npy", mmap_mode="r")
        self.indices = (
            np.arange(len(self.source_offsets) - 1) if indices is None else np.asarray(indices)
        )
        self.lengths = (
            np.diff(self.source_offsets)[self.indices] + np.diff(self.target_offsets)[self.indices]
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = self.indices[index]
        source = self.source[self.source_offsets[row] : self.source_offsets[row + 1]]
        target = self.target[self.target_offsets[row] : self.target_offsets[row + 1]]
        return source.astype(np.int64), target.astype(np.int64), index


def collate(rows):
    source_width = math.ceil(max(len(row[0]) for row in rows) / 8) * 8
    target_width = math.ceil(max(len(row[1]) for row in rows) / 8) * 8
    source = torch.full((len(rows), source_width), PAD, dtype=torch.long)
    labels = torch.full((len(rows), target_width), -100, dtype=torch.long)
    for index, (src, target, _) in enumerate(rows):
        source[index, : len(src)] = torch.from_numpy(src)
        labels[index, : len(target)] = torch.from_numpy(target)
    decoder = torch.full_like(labels, PAD)
    decoder[:, 0] = BOS
    decoder[:, 1:] = labels[:, :-1].masked_fill(labels[:, :-1] == -100, PAD)
    return {
        "input_ids": source,
        "attention_mask": source.ne(PAD),
        "labels": labels,
        "decoder_input_ids": decoder,
        "decoder_attention_mask": decoder.ne(PAD),
        "_indices": np.array([row[2] for row in rows], dtype=np.int64),
    }


"""Shuffle numeric indices, not millions of Python examples; 256-row update blocks."""
import hashlib
import json
import math
from pathlib import Path
import numpy as np


class UpdateOrder:
    def __init__(self, dataset, seed, epoch, effective=256):
        self.effective = effective
        rng = np.random.default_rng(seed + epoch)
        self.order = rng.permutation(len(dataset))
        width = effective * 50
        for start in range(0, len(dataset), width):
            chunk = self.order[start : start + width]
            self.order[start : start + width] = chunk[
                np.argsort(dataset.lengths[chunk], kind="stable")
            ]
        self.blocks = rng.permutation(math.ceil(len(dataset) / effective))

    def __len__(self):
        return len(self.blocks)

    def block(self, position):
        start = int(self.blocks[position]) * self.effective
        return self.order[start : start + self.effective]


def microbatches(dataset, indices, physical):
    for start in range(0, len(indices), physical):
        yield collate([dataset[int(i)] for i in indices[start : start + physical]])


def token_count(dataset, indices):
    rows = dataset.indices[indices]
    return int((dataset.target_offsets[rows + 1] - dataset.target_offsets[rows]).sum())


def verify_data(root):
    root = Path(root)
    m = verify_tree(root)
    groups = np.load(root / "train_groups.npy", mmap_mode="r")
    cache = root / "cache"
    train = TokenDataset(cache)
    val = TokenDataset(cache, "validation")
    if groups.shape != (len(train), 32) or len(train) != 5820047 or len(val) != 1500:
        raise ValueError("Frozen dataset dimensions changed")
    # Compute big-endian SHA256 modulo200 in bounded numpy chunks, without bigint lists.
    for start in range(0, len(groups), 65536):
        g = groups[start : start + 65536]
        r = np.zeros(len(g), dtype=np.uint32)
        for col in range(32):
            r = (r * 256 + g[:, col]) % 200
        if np.any(r == 0):
            raise ValueError("Reserved validation hash group in train")
    reserved = json.loads((root / "evaluation/validation_groups.json").read_text())
    if any(int(g, 16) % 200 != 0 for g in reserved):
        raise ValueError("Unexpected validation split")
    for kind, limit in [("source", 192), ("target", 384)]:
        offsets = getattr(train, kind + "_offsets")
        lengths = np.diff(offsets)
        if (
            offsets[0] != 0
            or offsets[-1] != len(getattr(train, kind))
            or np.any(lengths <= 0)
            or lengths.max() > limit
        ):
            raise ValueError("Bad token cache")
    questions = json.loads((root / "evaluation/questions_512.json").read_text())
    refs = json.loads((root / "evaluation/references_512.json").read_text())
    if len(questions) != 512 or len(refs) != 512 or sum(x["status"] == "ok" for x in refs) != 493:
        raise ValueError("Frozen generation set changed")
    return train, val, m, sha(root / "manifest.json")


if RUN_PHASES:
    BUDGET.require(180)
    if sha(DATA_ARCHIVE) != DATA_ARCHIVE_SHA256:
        raise IOError("Wrong/corrupted data ZIP")
    t = time.perf_counter()
    local_zip = LOCAL_ROOT / "colab_training_data.zip"
    publish_file(DATA_ARCHIVE, local_zip)
    extract_zip(local_zip, LOCAL_ROOT)
    DATA = LOCAL_ROOT / "colab_training_data"
    if sha(DATA / "manifest.json") != DATA_MANIFEST_SHA256:
        raise IOError("Frozen manifest changed")
    TRAIN, VAL, DATA_MANIFEST, DATA_ID = verify_data(DATA)
    TOKENIZER = Tokenizer.from_file(str(DATA / "tokenizer.json"))
    if [TOKENIZER.token_to_id(t) for t in ["<pad>", "<bos>", "<eos>", "<unk>"]] != [0, 1, 2, 3]:
        raise ValueError("Special tokens changed")
    if TOKENIZER.get_vocab_size() != 8192 or sha(DATA / "cache/tokenizer_8192.json") != sha(
        DATA / "tokenizer.json"
    ):
        raise ValueError("Tokenizer mismatch")
    QUESTIONS = json.loads((DATA / "evaluation/questions_512.json").read_text())
    IDENTITY = {
        "data_manifest_sha256": DATA_ID,
        "tokenizer_sha256": sha(DATA / "tokenizer.json"),
        "validation_questions_sha256": sha(DATA / "evaluation/questions_512.json"),
        "references_sha256": sha(DATA / "evaluation/references_512.json"),
        "test_sha256": sha(DATA / "test_questions.jsonl"),
        "implementation_sha256": IMPLEMENTATION_SHA256,
        "notebook_revision": NOTEBOOK_REVISION,
    }
    write_json(RUN / "configs/data_identity.json", IDENTITY)
    print(
        "mmap train:",
        len(TRAIN),
        "validation:",
        len(VAL),
        "group overlap0; data seconds:",
        time.perf_counter() - t,
    )
    print(
        "Available local GiB:",
        shutil.disk_usage(LOCAL_ROOT).free / 2**30,
        "Recommend20GiB initially; Drive10GiB free plus existing data",
    )


## 5. Модель, один эффективный update, checkpoint/resume
Случайная BART6+6,d768,12heads,FFN3072; shared embedding/LM head, PAD/BOS/EOS0/1/2, dropout0.1, позиции512. BF16 autocast без GradScaler, SDPA, AdamW, clipping1.0.

В update на256 примеров каждый loss умножается на число непустых target-токенов микробатча и делится на суммарные target-токены update. Это корректно для разных длин и неполного последнего блока. Перед abort частичного блока RNG откатывается; sampler продвигается только после успешного optimizer update.

In [ ]:
def create_model(config):
    cfg = BartConfig.from_dict(config)
    cfg._attn_implementation = "sdpa"
    return BartForConditionalGeneration(cfg)


"""Token-normalized effective updates, deterministic resume, BF16 without scaler."""
import contextlib
import copy
import math
import random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader


def seed_all(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)


def rng_state(device):
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
    }


def restore_rng(r):
    random.setstate(r["python"])
    np.random.set_state(r["numpy"])
    torch.set_rng_state(r["torch"])
    if r["cuda"] is not None:
        torch.cuda.set_rng_state_all(r["cuda"])


def autocast(device, dtype):
    return (
        torch.autocast(device.type, dtype=dtype) if dtype is not None else contextlib.nullcontext()
    )


def optimizer(model, lr, device):
    return torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.01,
        fused=device.type == "cuda",
    )


def scheduler(opt, planned, peak_policy="cosine", warmup=None):
    warmup = max(1, round(planned * 0.03)) if warmup is None else warmup

    def factor(step):
        if step < warmup:
            return max(1, step) / warmup
        if peak_policy == "constant_after_warmup":
            return 1.0
        x = min(1.0, (step - warmup) / max(1, planned - warmup))
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * x))

    return torch.optim.lr_scheduler.LambdaLR(opt, factor)


def backward_block(model, dataset, indices, physical, device, dtype, deadline=None):
    total = token_count(dataset, indices)
    if total <= 0:
        raise ValueError("No nonpadding target tokens")
    weighted_loss = 0.0
    source_tokens = 0
    model.train()
    for batch in microbatches(dataset, indices, physical):
        if deadline is not None:
            deadline.require()
        batch.pop("_indices")
        n = int(batch["labels"].ne(-100).sum())
        source_tokens += int(batch["attention_mask"].sum())
        batch = {k: v.to(device) for k, v in batch.items()}
        with autocast(device, dtype):
            loss = model(**batch, use_cache=False).loss
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError("Nonfinite training loss")
        # Each HF loss is a token mean. Normalize the SUM by all tokens in this update,
        # including unequal microbatches and an incomplete last accumulation block.
        (loss * (n / total)).backward()
        weighted_loss += float(loss.detach()) * n
    return {
        "loss": weighted_loss / total,
        "target_tokens": total,
        "source_tokens": source_tokens,
        "examples": len(indices),
    }


def update(model, opt, sched, dataset, indices, physical, device, dtype, deadline=None):
    before = rng_state(device)
    opt.zero_grad(set_to_none=True)
    try:
        result = backward_block(model, dataset, indices, physical, device, dtype, deadline)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not bool(torch.isfinite(norm)):
            raise FloatingPointError("Nonfinite gradients")
        if deadline is not None:
            deadline.require()
        opt.step()
        sched.step()
        return {**result, "gradient_norm": float(norm), "lr": opt.param_groups[0]["lr"]}
    except BaseException:
        opt.zero_grad(set_to_none=True)
        restore_rng(before)
        raise


def validation_ce(model, dataset, batch_size, device, dtype, deadline=None):
    r = rng_state(device)
    total = 0.0
    n = 0
    try:
        model.eval()
        for batch in DataLoader(dataset, batch_size=batch_size, collate_fn=collate):
            if deadline is not None:
                deadline.require(final=True)
            batch.pop("_indices")
            tokens = int(batch["labels"].ne(-100).sum())
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.inference_mode(), autocast(device, dtype):
                loss = model(**batch, use_cache=False).loss
            total += float(loss) * tokens
            n += tokens
        if not math.isfinite(total):
            raise FloatingPointError("Nonfinite validation CE")
        return {
            "validation_ce": total / n,
            "target_tokens": n,
            "examples": len(dataset),
            "batch_size": batch_size,
        }
    finally:
        restore_rng(r)


def save_checkpoint(path, model, opt, sched, state, config, identity, device, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".tmp.pt")
    payload = {
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": sched.state_dict(),
        "rng": rng_state(device),
        "state": copy.deepcopy(state),
        "config": copy.deepcopy(config),
        "identity": identity,
        "extra": extra or {},
        "checkpoint_format": "colab_effective_token_v1",
        "precision": "bf16_no_scaler" if device.type == "cuda" else "float32_test",
    }
    torch.save(payload, temp)
    temp.replace(path)


def restore_checkpoint(path, model, opt, sched, identity, config, device):
    c = torch.load(path, map_location="cpu", weights_only=False)
    if c["checkpoint_format"] != "colab_effective_token_v1" or c["identity"] != identity:
        raise ValueError("Checkpoint/data/code identity mismatch")
    # Never silently recalculate the schedule, batch geometry or model on resume.
    for key in [
        "model",
        "planned_steps",
        "warmup_steps",
        "peak_lr",
        "physical_batch",
        "effective_batch",
        "seed",
    ]:
        if c["config"][key] != config[key]:
            raise ValueError(f"Resume configuration changed: {key}")
    model.load_state_dict(c["model"])
    opt.load_state_dict(c["optimizer"])
    sched.load_state_dict(c["scheduler"])
    restore_rng(c["rng"])
    return c["state"], c["extra"]


STOP_REQUESTED = False


def stop_requested(signum, frame):
    global STOP_REQUESTED
    STOP_REQUESTED = True


class PhaseBudget:
    def __init__(self, parent, end):
        self.parent = parent
        self.end = min(end, parent.cutoff)

    def require(self, seconds=0, final=False):
        self.parent.require(seconds, final=final)
        if STOP_REQUESTED or time.time() + seconds >= self.end:
            raise DeadlineReached("Phase deadline / stop request")


if RUN_PHASES:
    signal.signal(signal.SIGTERM, stop_requested)
    signal.signal(signal.SIGINT, stop_requested)
    BUDGET.require(180)
    INITIAL = RUN / "checkpoints/initial_weights.pt"
    INITIAL_RNG = None
    if RESUME_LOCAL is None:
        seed_all(CONFIG["seed"], DEVICE)
        m = create_model(CONFIG["model"])
        PARAMETERS = sum(p.numel() for p in m.parameters())
        if PARAMETERS != 106321920:
            raise ValueError("Unexpected architecture")
        torch.save(m.state_dict(), INITIAL)
        del m
        INITIAL_RNG = rng_state(DEVICE)
        INITIAL_SHA = sha(INITIAL)
    else:
        c = torch.load(RESUME_LOCAL, map_location="cpu", weights_only=False)
        if c["identity"] != IDENTITY:
            raise ValueError("Resume identity differs; do not silently change code/data")
        RUNTIME_CONFIG = c["config"]
        PARAMETERS = sum(p.numel() for p in create_model(RUNTIME_CONFIG["model"]).parameters())
        del c
        INITIAL_SHA = "resume retains original random initialization identity"
    print("Unique model parameters:", PARAMETERS, "No pretrained weights loaded.")
    write_json(RUN / "configs/requested.json", CONFIG)


def fresh_model(lr):
    model = create_model(CONFIG["model"]).to(DEVICE)
    weights = torch.load(INITIAL, map_location="cpu", weights_only=True)
    model.load_state_dict(weights)
    del weights
    opt = optimizer(model, lr, DEVICE)
    restore_rng(INITIAL_RNG)
    return model, opt


## 6. Benchmark physical batch32/64/128/256
Effective batch всегда256. Два warmup update, затем реальные блоки разной длины и один из самых длинных. Измеряются полезные target-токены/с, примеры/с, время update и allocated/reserved. Для прогноза используются обычные блоки; длинный блок — стресс-проверка памяти. Benchmark-модели выбрасываются, общая начальная инициализация проб не меняется.

In [ ]:
ABORT_REASON = None
BENCHMARK = None
TRIALS = []
CANDIDATES = []
MAIN_MODEL = None
MAIN_OPT = None
MAIN_SCHED = None
MAIN_STATE = None


def log_event(kind, **values):
    row = {
        "event": kind,
        "time_unix": time.time(),
        "a100_elapsed_seconds": BUDGET.elapsed(),
        **values,
    }
    with (RUN / "logs/events.jsonl").open("a", buffering=1) as f:
        f.write(json.dumps(row, allow_nan=False) + "\n")
    if kind != "train" or values.get("updates", 0) % 1000 == 0:
        print(row, flush=True)


def sync_small():
    try:
        for folder in ["configs", "logs", "validation_predictions"]:
            for p in (RUN / folder).rglob("*"):
                if p.is_file():
                    publish_file(p, DRIVE_RUN / p.relative_to(RUN))
    except Exception as exc:
        drive_failure(exc)
        raise


def clean_cuda():
    gc.collect()
    torch.cuda.empty_cache()


def benchmark_batches():
    end = min(
        time.time() + CONFIG["budget"]["benchmark_seconds"],
        SESSION_STARTED_UNIX + CONFIG["budget"]["prepare_seconds"],
        BUDGET.cutoff - 600,
    )
    phase = PhaseBudget(BUDGET, end)
    order = UpdateOrder(TRAIN, CONFIG["seed"], 0)
    blocks = [order.block(i) for i in range(3)]
    longest = np.argpartition(TRAIN.lengths, -256)[-256:]
    measured = blocks + [longest]
    results = []
    validation_seconds = None
    for physical in CONFIG["training"]["physical_batch_candidates"]:
        phase.require(45)
        model = opt = sched = None
        try:
            model, opt = fresh_model(0.0002)
            sched = scheduler(opt, 100, "constant_after_warmup", warmup=1)
            for block in [blocks[0], longest]:
                update(model, opt, sched, TRAIN, block, physical, DEVICE, DTYPE, phase)
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            times = []
            nt = ne = 0
            for block in measured:
                phase.require(30)
                torch.cuda.synchronize()
                t = time.perf_counter()
                r = update(model, opt, sched, TRAIN, block, physical, DEVICE, DTYPE, phase)
                torch.cuda.synchronize()
                times.append(time.perf_counter() - t)
                nt += r["target_tokens"]
                ne += r["examples"]
            memory = torch.cuda.get_device_properties(0).total_memory
            safe = torch.cuda.max_memory_reserved() < 0.8 * memory
            row = {
                "physical_batch": physical,
                "accumulation": 256 // physical,
                "effective_batch": 256,
                "ok": safe,
                "examples_per_second": ne / sum(times),
                "useful_target_tokens_per_second": nt / sum(times),
                "seconds_per_update": sum(times) / len(times),
                "typical_update_seconds": sum(times[:3]) / 3,
                "long_update_seconds": times[-1],
                "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
                "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
                "memory_margin_fraction": 0.2,
                "all_loss_gradients_finite": True,
            }
            if validation_seconds is None:
                t = time.perf_counter()
                validation_ce(model, VAL, 32, DEVICE, DTYPE, phase)
                torch.cuda.synchronize()
                validation_seconds = time.perf_counter() - t
            results.append(row)
        except torch.cuda.OutOfMemoryError:
            results.append(
                {
                    "physical_batch": physical,
                    "effective_batch": 256,
                    "ok": False,
                    "reason": "CUDA OOM; model/optimizer discarded",
                }
            )
        finally:
            del model, opt, sched
            clean_cuda()
        write_json(
            RUN / "logs/benchmark.json", {"rows": results, "validation_seconds": validation_seconds}
        )
    good = [r for r in results if r["ok"]]
    if not good:
        raise RuntimeError("No stable physical batch with20% memory margin; no uncontrolled search")
    chosen = max(good, key=lambda r: r["useful_target_tokens_per_second"])
    estimate = {
        "epoch_seconds_estimate": math.ceil(len(TRAIN) / 256) * chosen["typical_update_seconds"],
        "examples_in_4_5h_before_evaluation_overheads_estimate": int(
            16200 / chosen["typical_update_seconds"]
        )
        * 256,
        "not_a_guarantee": True,
    }
    return {
        "rows": results,
        "chosen": chosen,
        "validation_seconds": validation_seconds,
        "forecast": estimate,
        "benchmark_models_discarded": True,
        "initial_weight_sha256": INITIAL_SHA,
    }


if RUN_PHASES:
    if RESUME_LOCAL is None:
        try:
            BENCHMARK = benchmark_batches()
            write_json(RUN / "logs/benchmark.json", BENCHMARK)
            sync_small()
            print("Chosen batch and forecast:", BENCHMARK["chosen"], BENCHMARK["forecast"])
        except Exception as exc:
            ABORT_REASON = "benchmark: " + repr(exc)
            log_event("abort", reason=ABORT_REASON)
    else:
        print("Resume: benchmark and LR search skipped; saved schedule and batch retained")
        for folder in ["configs", "logs"]:
            if (DRIVE_RUN / folder).exists():
                for p in (DRIVE_RUN / folder).glob("*"):
                    if p.is_file():
                        shutil.copy2(p, RUN / folder / p.name)


## 7. Две LR-пробы:2e-4 и5e-4
Одинаковые случайные веса и RNG, seed, effective256, число updates, sampler и validation. Короткий warmup, затем постоянный peak LR без затухания. Выбор: среди завершивших все общие шаги и численно стабильных проб — меньший token-weighted CE. Одна нестабильная исключается; если подходящих нет, main не запускается. Короткое сравнение не предсказывает гарантированно лучший LR на длинной траектории.

In [ ]:
def generate_records(
    model, tokenizer, texts, batch_size=32, max_new_tokens=256, device=None, deadline=None
):
    device = next(model.parameters()).device if device is None else device
    model.eval()
    records = []
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for offset in range(0, len(texts), batch_size):
        if deadline is not None:
            deadline.require(30, final=True)
        encoded = [
            [1] + x.ids + [2] for x in tokenizer.encode_batch(texts[offset : offset + batch_size])
        ]
        if max(map(len, encoded)) > 192:
            raise ValueError("Source exceeds192; no silent truncation")
        width = math.ceil(max(map(len, encoded)) / 8) * 8
        ids = torch.zeros((len(encoded), width), dtype=torch.long, device=device)
        for i, row in enumerate(encoded):
            ids[i, : len(row)] = torch.tensor(row, device=device)
        with torch.inference_mode(), autocast(
            device, torch.float16 if device.type == "cuda" else None
        ):
            out = model.generate(
                input_ids=ids,
                attention_mask=ids.ne(0),
                do_sample=False,
                num_beams=1,
                use_cache=True,
                max_new_tokens=max_new_tokens,
                forced_eos_token_id=None,
                pad_token_id=0,
                eos_token_id=2,
                decoder_start_token_id=1,
            )
        for i, row in enumerate(out.cpu().tolist()):
            row = row[1:]
            eos = 2 in row
            if eos:
                row = row[: row.index(2) + 1]
            records.append(
                {
                    "index": offset + i,
                    "question_sha256": hashlib.sha256(texts[offset + i].encode()).hexdigest(),
                    "code": tokenizer.decode(row, skip_special_tokens=True),
                    "tokens": len(row),
                    "eos": eos,
                    "hit_limit": not eos and len(row) >= max_new_tokens,
                }
            )
    if device.type == "cuda":
        torch.cuda.synchronize()
    return records, {
        "generation_seconds": time.perf_counter() - started,
        "examples": len(records),
        "batch_size": batch_size,
        "max_new_tokens": max_new_tokens,
        "precision": "float16" if device.type == "cuda" else "float32_test",
        "warmup_excluded": False,
    }


def save_generations(model, name, texts, deadline):
    rng = rng_state(DEVICE)
    try:
        predictions, timing = generate_records(model, TOKENIZER, texts, 32, 256, DEVICE, deadline)
        write_json(
            RUN / "validation_predictions" / (name + ".json"),
            {
                "predictions": predictions,
                "timing": timing,
                "identity": IDENTITY,
                "question_count": len(texts),
                "numerical_validation_accuracy": None,
                "public_pass_at_1": None,
                "generated_programs_executed": False,
            },
        )
        return timing
    finally:
        restore_rng(rng)


def lr_trials():
    start = time.time()
    end = min(start + 2400, BUDGET.cutoff - 300)
    ce_cost = BENCHMARK["validation_seconds"]
    sec = BENCHMARK["chosen"]["typical_update_seconds"]
    common = max(1, int(((end - start) - 2 * (ce_cost + 150)) * 0.8 / (2 * sec)))
    warmup = max(1, min(100, math.ceil(common * 0.05)))
    write_json(
        RUN / "configs/lr_trials.json",
        {
            "peak_lrs": [0.0002, 0.0005],
            "common_updates": common,
            "warmup_updates": warmup,
            "after_warmup": "constant",
            "initial_weight_sha256": INITIAL_SHA,
            "effective_batch": 256,
            "selection_rule": "minimum CE among finite completed common-update trials; no numerical execution in Colab",
            "limitations": "short horizon, one seed, no mathematical accuracy measured",
        },
    )
    results = []
    for number, lr in enumerate([0.0002, 0.0005]):
        trial_end = start + (number + 1) * (end - start) / 2
        phase = PhaseBudget(BUDGET, trial_end - 150)
        model = opt = sched = None
        row = {
            "peak_lr": lr,
            "planned_updates": common,
            "completed_updates": 0,
            "stable": False,
            "initial_weight_sha256": INITIAL_SHA,
            "loss_samples": [],
        }
        try:
            model, opt = fresh_model(lr)
            sched = scheduler(opt, common, "constant_after_warmup", warmup)
            epoch = pos = 0
            order = UpdateOrder(TRAIN, CONFIG["seed"], epoch)
            for step in range(common):
                phase.require()
                r = update(
                    model,
                    opt,
                    sched,
                    TRAIN,
                    order.block(pos),
                    BENCHMARK["chosen"]["physical_batch"],
                    DEVICE,
                    DTYPE,
                    phase,
                )
                pos += 1
                if pos == len(order):
                    epoch += 1
                    pos = 0
                    order = UpdateOrder(TRAIN, CONFIG["seed"], epoch)
                row["completed_updates"] += 1
                if step % 25 == 0:
                    row["loss_samples"].append({"step": step + 1, **r})
                    (
                        log_event("lr_trial", lr=lr, updates=step + 1, loss=r["loss"])
                        if step % 1000 == 0
                        else None
                    )
            evaluation_budget = PhaseBudget(BUDGET, trial_end)
            ce = validation_ce(
                model, VAL, CONFIG["training"]["ce_batch_size"], DEVICE, DTYPE, evaluation_budget
            )
            row.update(ce)
            row["sample_generation"] = save_generations(
                model, f"lr_{lr}_sample32", QUESTIONS[:32], evaluation_budget
            )
            row["stable"] = True
        except (DeadlineReached, FloatingPointError, torch.cuda.OutOfMemoryError) as exc:
            row["error"] = repr(exc)
        finally:
            results.append(row)
            write_json(RUN / "logs/lr_trials.json", results)
            del model, opt, sched
            clean_cuda()
        sync_small()
    return results


if RUN_PHASES:
    if RESUME_LOCAL is None and ABORT_REASON is None:
        try:
            TRIALS = lr_trials()
            eligible = [
                r for r in TRIALS if r["stable"] and r["completed_updates"] == r["planned_updates"]
            ]
            if not eligible:
                raise RuntimeError(
                    "Both LR trials unstable/incomplete; stop, do not start new recipes"
                )
            SELECTED_LR = min(eligible, key=lambda r: r["validation_ce"])["peak_lr"]
            write_json(
                RUN / "configs/lr_choice.json",
                {
                    "selected_peak_lr": SELECTED_LR,
                    "criterion": "token-weighted CE; numerical validation/public unknown",
                    "eligible_lrs": [r["peak_lr"] for r in eligible],
                },
            )
            log_event("lr_choice", peak_lr=SELECTED_LR)
        except Exception as exc:
            ABORT_REASON = "LR comparison: " + repr(exc)
            log_event("abort", reason=ABORT_REASON)
    elif RESUME_LOCAL is not None:
        SELECTED_LR = RUNTIME_CONFIG["peak_lr"]


## 8. Основное обучение с исходной случайной инициализации
LR-проба не продолжается. Planned steps и warmup3% фиксируются один раз, cosine до0.1 peak. CE примерно раз в25 минут, полная greedy validation примерно раз в час при наличии времени. До трёх лучших весов по CE сохраняются отдельно от полного `last`. Раннее отставание от старого baseline не является stop condition. Полный state и модель сохраняются локально до копирования на Drive.

In [ ]:
def save_last(model, opt, sched, state):
    BUDGET.require(30, final=True)
    config = RUNTIME_CONFIG
    path = RUN / "checkpoints/last.pt"
    save_checkpoint(
        path,
        model,
        opt,
        sched,
        state,
        config,
        IDENTITY,
        DEVICE,
        {"candidates": CANDIDATES, "last_ce": state.get("last_ce")},
    )
    try:
        pointer = publish_checkpoint(
            path,
            DRIVE_RUN / "checkpoints",
            {
                "updates": state["updates"],
                "a100_elapsed_seconds": BUDGET.elapsed(),
                "session_started_unix": BUDGET.start,
                "prior_elapsed_seconds": BUDGET.prior,
                "identity": IDENTITY,
                "planned_steps": config["planned_steps"],
                "physical_batch": config["physical_batch"],
            },
        )
        log_event("checkpoint", updates=state["updates"], sha256=pointer["sha256"])
    except Exception as exc:
        drive_failure(exc)
        raise


def add_ce_candidate(model, ce, step):
    global CANDIDATES
    score = ce["validation_ce"]
    maximum = CONFIG["training"]["max_candidates"]
    if len(CANDIDATES) >= maximum and score >= max(r["ce"] for r in CANDIDATES):
        return
    name = f"candidate_{step:09d}"
    local = RUN / "exports" / name
    if local.exists():
        return
    local.mkdir(parents=True)
    model.save_pretrained(local / "model", safe_serialization=True)
    record = {
        "name": name,
        "ce": score,
        "update": step,
        "selection": "provisional CE only",
        "numerical_accuracy": None,
    }
    write_json(local / "candidate.json", record)
    try:
        publish_tree(local, DRIVE_RUN / "exports" / name)
    except Exception as exc:
        drive_failure(exc)
        raise
    candidates = sorted(CANDIDATES + [record], key=lambda r: r["ce"])
    keep = candidates[:maximum]
    remove = candidates[maximum:]
    # Advertise newly verified candidate BEFORE removing any obsolete weights.
    write_json(RUN / "configs/candidates.json", keep)
    publish_file(RUN / "configs/candidates.json", DRIVE_RUN / "configs/candidates.json")
    CANDIDATES = keep
    for old in remove:
        for root in [RUN, DRIVE_RUN]:
            folder = root / "exports" / old["name"]
            if folder.exists():
                shutil.rmtree(folder)


def main_fit():
    global MAIN_MODEL, MAIN_OPT, MAIN_SCHED, MAIN_STATE, RUNTIME_CONFIG, CANDIDATES
    if RESUME_LOCAL is None:
        available = min(CONFIG["budget"]["main_seconds"], BUDGET.remaining() - 90)
        if available < 180:
            raise DeadlineReached("Not enough main budget")
        ce_cost = BENCHMARK["validation_seconds"]
        anticipated = (
            math.ceil(available / 1500) * (ce_cost + 75) + math.ceil(available / 3600) * 120 + 120
        )
        overhead = max(0.12 * available, anticipated)
        planned = max(
            1, int((available - overhead) / BENCHMARK["chosen"]["typical_update_seconds"])
        )
        RUNTIME_CONFIG = {
            "model": CONFIG["model"],
            "seed": CONFIG["seed"],
            "effective_batch": 256,
            "physical_batch": BENCHMARK["chosen"]["physical_batch"],
            "peak_lr": SELECTED_LR,
            "planned_steps": planned,
            "warmup_steps": max(1, round(planned * 0.03)),
            "cosine_floor": 0.1,
            "generation": CONFIG["generation"],
            "initial_weight_sha256": INITIAL_SHA,
            "estimated_evaluation_save_overhead_seconds": overhead,
            "main_wall_cap_seconds": available,
        }
        MAIN_MODEL, MAIN_OPT = fresh_model(SELECTED_LR)
        MAIN_SCHED = scheduler(MAIN_OPT, planned, warmup=RUNTIME_CONFIG["warmup_steps"])
        MAIN_STATE = {
            "updates": 0,
            "examples": 0,
            "target_tokens": 0,
            "source_tokens": 0,
            "sampler_epoch": 0,
            "update_in_epoch": 0,
        }
    else:
        MAIN_MODEL = create_model(RUNTIME_CONFIG["model"]).to(DEVICE)
        MAIN_OPT = optimizer(MAIN_MODEL, RUNTIME_CONFIG["peak_lr"], DEVICE)
        MAIN_SCHED = scheduler(
            MAIN_OPT, RUNTIME_CONFIG["planned_steps"], warmup=RUNTIME_CONFIG["warmup_steps"]
        )
        MAIN_STATE, extra = restore_checkpoint(
            RESUME_LOCAL, MAIN_MODEL, MAIN_OPT, MAIN_SCHED, IDENTITY, RUNTIME_CONFIG, DEVICE
        )
        path = DRIVE_RUN / "configs/candidates.json"
        CANDIDATES = json.loads(path.read_text()) if path.exists() else extra.get("candidates", [])
        for candidate in CANDIDATES:
            if not (DRIVE_RUN / "exports" / candidate["name"] / "model/model.safetensors").exists():
                raise IOError("Saved candidate missing; do not silently drop history")
    write_json(RUN / "configs/runtime.json", RUNTIME_CONFIG)
    sync_small()
    main_end = min(time.time() + CONFIG["budget"]["main_seconds"], BUDGET.cutoff - 120)
    phase = PhaseBudget(BUDGET, main_end)
    log_event("main_start", config=RUNTIME_CONFIG, resume=RESUME_LOCAL is not None)
    save_last(MAIN_MODEL, MAIN_OPT, MAIN_SCHED, MAIN_STATE)
    next_ce = time.time() + 1500
    next_gen = time.time() + 3600
    model_boundary_valid = True
    error = None
    try:
        while MAIN_STATE["updates"] < RUNTIME_CONFIG["planned_steps"]:
            phase.require(90)
            order = UpdateOrder(TRAIN, RUNTIME_CONFIG["seed"], MAIN_STATE["sampler_epoch"])
            while (
                MAIN_STATE["update_in_epoch"] < len(order)
                and MAIN_STATE["updates"] < RUNTIME_CONFIG["planned_steps"]
            ):
                phase.require(90)
                indices = order.block(MAIN_STATE["update_in_epoch"])
                model_boundary_valid = False
                row = update(
                    MAIN_MODEL,
                    MAIN_OPT,
                    MAIN_SCHED,
                    TRAIN,
                    indices,
                    RUNTIME_CONFIG["physical_batch"],
                    DEVICE,
                    DTYPE,
                    phase,
                )
                model_boundary_valid = True
                MAIN_STATE["updates"] += 1
                MAIN_STATE["update_in_epoch"] += 1
                for key in ["examples", "target_tokens", "source_tokens"]:
                    MAIN_STATE[key] += row[key]
                if MAIN_STATE["updates"] % 25 == 0:
                    log_event(
                        "train",
                        **MAIN_STATE,
                        loss=row["loss"],
                        lr=row["lr"],
                        gradient_norm=row["gradient_norm"],
                        allocated_gib=torch.cuda.memory_allocated() / 2**30,
                        reserved_gib=torch.cuda.memory_reserved() / 2**30,
                    )
                if time.time() >= next_ce and phase.end - time.time() > max(
                    180, 2 * (BENCHMARK["validation_seconds"] if BENCHMARK else 30) + 120
                ):
                    save_last(MAIN_MODEL, MAIN_OPT, MAIN_SCHED, MAIN_STATE)
                    ce = validation_ce(MAIN_MODEL, VAL, 32, DEVICE, DTYPE, phase)
                    MAIN_STATE["last_ce"] = ce["validation_ce"]
                    log_event("validation_ce", updates=MAIN_STATE["updates"], **ce)
                    add_ce_candidate(MAIN_MODEL, ce, MAIN_STATE["updates"])
                    sync_small()
                    next_ce = time.time() + 1500
                if time.time() >= next_gen and phase.end - time.time() > 300:
                    save_last(MAIN_MODEL, MAIN_OPT, MAIN_SCHED, MAIN_STATE)
                    save_generations(
                        MAIN_MODEL, f'periodic_{MAIN_STATE["updates"]:09d}_512', QUESTIONS, phase
                    )
                    sync_small()
                    next_gen = time.time() + 3600
                if (
                    torch.cuda.memory_reserved()
                    > 0.85 * torch.cuda.get_device_properties(0).total_memory
                ):
                    torch.cuda.empty_cache()
            if MAIN_STATE["update_in_epoch"] >= len(order):
                MAIN_STATE["sampler_epoch"] += 1
                MAIN_STATE["update_in_epoch"] = 0
    except DeadlineReached as exc:
        # update() restores RNG and does not step after a deadline; current boundary is valid.
        model_boundary_valid = True
        log_event("training_deadline", reason=str(exc))
    except Exception as exc:
        error = repr(exc)
        log_event("main_error", reason=error)
    finally:
        if model_boundary_valid:
            save_last(MAIN_MODEL, MAIN_OPT, MAIN_SCHED, MAIN_STATE)
        else:
            # A failed fused optimizer may have partially mutated tensors. Reload known-good
            # local state rather than advertise such weights as a successful checkpoint.
            good = torch.load(RUN / "checkpoints/last.pt", map_location="cpu", weights_only=False)
            MAIN_MODEL.load_state_dict(good["model"])
            MAIN_OPT.load_state_dict(good["optimizer"])
            MAIN_SCHED.load_state_dict(good["scheduler"])
            restore_rng(good["rng"])
            MAIN_STATE = good["state"]
            del good
            log_event("recovered_local_boundary", updates=MAIN_STATE["updates"])
        write_json(
            RUN / "logs/main_result.json",
            {
                "state": MAIN_STATE,
                "error": error,
                "schedule_unchanged": True,
                "numerical_accuracy": None,
                "public_pass_at_1": None,
            },
        )
    return error


if RUN_PHASES and ABORT_REASON is None:
    try:
        ABORT_REASON = main_fit()
    except Exception as exc:
        ABORT_REASON = "main/final-save: " + repr(exc)
        log_event("abort", reason=ABORT_REASON)


## 9. Финальное сохранение
После этой ячейки updates не выполняются. Используется последний30-минутный резерв; при малом остатке длинные оценки не запускаются. Ошибка Drive видна в выводе и локальном `drive_error.json`; исправный remote checkpoint не удаляется до проверки следующего. Это не гарантия против внезапного уничтожения VM.

In [ ]:
FINAL_CE = None
if RUN_PHASES:
    print("TRAINING UPDATES FINISHED. Remaining final seconds:", BUDGET.remaining(True))
    if MAIN_MODEL is not None and DRIVE_ERROR is None:
        try:
            if BUDGET.remaining(True) > 180:
                FINAL_CE = validation_ce(MAIN_MODEL, VAL, 32, DEVICE, DTYPE, BUDGET)
                MAIN_STATE["last_ce"] = FINAL_CE["validation_ce"]
                add_ce_candidate(MAIN_MODEL, FINAL_CE, MAIN_STATE["updates"])
            save_last(MAIN_MODEL, MAIN_OPT, MAIN_SCHED, MAIN_STATE)
            local_final = RUN / "exports/final"
            local_final.mkdir(exist_ok=False)
            MAIN_MODEL.save_pretrained(local_final / "model", safe_serialization=True)
            write_json(
                local_final / "candidate.json",
                {
                    "name": "final",
                    "ce": FINAL_CE["validation_ce"] if FINAL_CE else None,
                    "update": MAIN_STATE["updates"],
                    "selection": "final weights; not automatically mathematical best",
                },
            )
            publish_tree(local_final, DRIVE_RUN / "exports/final")
        except Exception as exc:
            drive_failure(exc)
            ABORT_REASON = "final save: " + repr(exc)
    MAIN_MODEL = MAIN_OPT = MAIN_SCHED = None
    clean_cuda()
    write_json(
        RUN / "logs/final_status.json",
        {
            "abort_reason": ABORT_REASON,
            "drive_error": DRIVE_ERROR,
            "main_state": MAIN_STATE,
            "final_ce": FINAL_CE,
            "numerical_accuracy": None,
            "public_pass_at_1": None,
        },
    )
    try:
        sync_small()
    except Exception:
        pass


## 10. Экспорт, generation кандидатов и проверка Drive
До трёх CE-кандидатов плюс final: собственные safetensors/config/tokenizer, greedy параметры, inference-код,1919 неизменных вопросов, manifests, без optimizer/train. Generated-код **не исполняется**. Полные validation-генерации передаются локальной Docker-песочнице. После проверки файлов на Drive и загрузки каждой модели появляется `completion.json`; только тогда выводится сообщение, что A100 можно отключать.

In [ ]:
INFERENCE_SOURCE = r"""import os, sys, time, json, hashlib, math, platform, importlib.metadata
from pathlib import Path

PIPELINE_STARTED = time.perf_counter()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
from transformers import BartForConditionalGeneration
from tokenizers import Tokenizer
from contextlib import nullcontext


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            h.update(chunk)
    return h.hexdigest()


def infer_generate(model, tokenizer, texts, batch_size=64, max_new_tokens=256):
    records = []
    model.eval()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for offset in range(0, len(texts), batch_size):
        if time.time() + 30 >= globals().get("INFERENCE_DEADLINE_UNIX", float("inf")):
            raise TimeoutError("T4 wall-clock budget reached")
        encoded = [
            [1] + x.ids + [2] for x in tokenizer.encode_batch(texts[offset : offset + batch_size])
        ]
        if max(map(len, encoded)) > 192:
            raise ValueError("Source exceeds192; no silent truncation")
        width = math.ceil(max(map(len, encoded)) / 8) * 8
        ids = torch.zeros((len(encoded), width), dtype=torch.long, device="cuda")
        for i, row in enumerate(encoded):
            ids[i, : len(row)] = torch.tensor(row, device="cuda")
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            outputs = model.generate(
                input_ids=ids,
                attention_mask=ids.ne(0),
                do_sample=False,
                num_beams=1,
                use_cache=True,
                max_new_tokens=max_new_tokens,
                forced_eos_token_id=None,
                pad_token_id=0,
                eos_token_id=2,
                decoder_start_token_id=1,
            )
        for i, row in enumerate(outputs.cpu().tolist()):
            row = row[1:]
            eos = 2 in row
            if eos:
                row = row[: row.index(2) + 1]
            records.append(
                {
                    "index": offset + i,
                    "question_sha256": hashlib.sha256(texts[offset + i].encode()).hexdigest(),
                    "code": tokenizer.decode(row, skip_special_tokens=True),
                    "tokens": len(row),
                    "eos": eos,
                    "hit_limit": not eos and len(row) >= max_new_tokens,
                }
            )
    torch.cuda.synchronize()
    return records, {
        "generation_seconds": time.perf_counter() - started,
        "examples": len(records),
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
        "batch_size": batch_size,
    }


def inference_run(bundle, output, batch_size=64, require_t4=True):
    bundle, output = Path(bundle), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Exactly one visible CUDA GPU required")
    gpu = torch.cuda.get_device_name(0)
    if require_t4 and "T4" not in gpu:
        raise RuntimeError("Select T4; this notebook is the T4 measurement")
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    versions = {
        n: importlib.metadata.version(n)
        for n in ["torch", "transformers", "tokenizers", "numpy", "safetensors"]
    }
    print(
        "GPU:",
        gpu,
        "CUDA:",
        torch.version.cuda,
        "Python:",
        sys.version,
        "versions:",
        versions,
        flush=True,
    )
    manifest = json.loads((bundle / "bundle_manifest.json").read_text())
    for name, digest in manifest["asset_sha256"].items():
        path = (bundle / name).resolve()
        if bundle.resolve() not in path.parents or file_sha(path) != digest:
            raise IOError("Export checksum mismatch: " + name)
    inputs = [json.loads(x) for x in (bundle / "test_questions.jsonl").read_text().splitlines()]
    if len(inputs) != 1919 or any(
        set(r) != {"index", "question"} or r["index"] != i or not isinstance(r["question"], str)
        for i, r in enumerate(inputs)
    ):
        raise ValueError("Expected1919 original indexed question-only rows")
    texts = [r["question"] for r in inputs]
    torch.cuda.synchronize()
    t = time.perf_counter()
    model = (
        BartForConditionalGeneration.from_pretrained(
            bundle / "model", local_files_only=True, attn_implementation="sdpa"
        )
        .cuda()
        .eval()
    )
    tokenizer = Tokenizer.from_file(str(bundle / "tokenizer.json"))
    if [tokenizer.token_to_id(s) for s in ["<pad>", "<bos>", "<eos>", "<unk>"]] != [0, 1, 2, 3]:
        raise ValueError("Special tokens differ")
    torch.cuda.synchronize()
    loading = time.perf_counter() - t
    _, warm = infer_generate(model, tokenizer, texts[:8], 8)
    torch.cuda.empty_cache()
    records, timing = infer_generate(model, tokenizer, texts, batch_size)
    if len(records) != 1919:
        raise ValueError("Output count differs")
    for i, (record, text) in enumerate(zip(records, texts)):
        if (
            record["index"] != i
            or record["question_sha256"] != hashlib.sha256(text.encode()).hexdigest()
        ):
            raise ValueError("Input/output alignment differs")
    (output / "predictions.json").write_text(json.dumps(records, ensure_ascii=False, indent=2))
    submission = output / "submission.jsonl"
    with submission.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps({"code": record["code"]}, ensure_ascii=False) + "\n")
    parsed = [json.loads(line) for line in submission.read_text().splitlines()]
    assert len(parsed) == 1919 and all(
        set(r) == {"code"} and r["code"] == x["code"] for r, x in zip(parsed, records)
    )
    settings = {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 256,
        "batch_size": batch_size,
        "precision": "float16",
        "sdpa": True,
        "kv_cache": True,
        "forced_eos_token_id": None,
    }
    (output / "generation_parameters.json").write_text(json.dumps(settings, indent=2))
    (output / "submission_validation.json").write_text(
        json.dumps(
            {
                "count": 1919,
                "only_code_key": True,
                "original_bundled_order_verified": True,
                "official_pass1_file_separately_compared": False,
                "input_sha256": file_sha(bundle / "test_questions.jsonl"),
                "submission_sha256": file_sha(submission),
                "programs_edited_or_executed": False,
                "bot_submission_sent": False,
            },
            indent=2,
        )
    )
    timing.update(
        {
            "gpu": gpu,
            "versions": versions,
            "load_model_tokenizer_seconds": loading,
            "warmup_seconds": warm["generation_seconds"],
            "warmup_examples": 8,
            "full_pipeline_seconds": time.perf_counter() - PIPELINE_STARTED,
            "pipeline_includes": "imports in this cell/process, checksums, CUDA initialization, model/tokenizer load, warmup, full generation, output writes up to timing.json",
            "pipeline_excludes": "Drive mount/copy and dependency installation; reported separately by notebook",
            "generation_under_180_seconds": timing["generation_seconds"] <= 180,
            "full_pipeline_under_180_seconds": time.perf_counter() - PIPELINE_STARTED <= 180,
            "T4_measured": "T4" in gpu,
            "public_pass_at_1": None,
            "identity": manifest["identity"],
        }
    )
    (output / "timing.json").write_text(json.dumps(timing, indent=2))
    print(json.dumps(timing, indent=2), flush=True)
    return timing


if __name__ == "__main__" and "IN_NOTEBOOK" not in globals():
    import argparse

    p = argparse.ArgumentParser()
    p.add_argument("--bundle", type=Path, required=True)
    p.add_argument("--output", type=Path, required=True)
    p.add_argument("--batch", type=int, default=64)
    a = p.parse_args()
    inference_run(a.bundle, a.output, a.batch)
"""
INFERENCE_REQUIREMENTS = """# Kaggle supplies CUDA-enabled torch; do not replace it automatically.
# Exact local reference torch:2.5.1+cu124. Other versions must be measured/reported.
transformers==4.46.3
tokenizers==0.20.3
safetensors>=0.4.3,<1
numpy>=1.26,<3
"""


def complete_portable_export(local, record):
    # Candidate weights were already saved; add only portable inference assets here.
    for name in ["tokenizer.json", "test_questions.jsonl"]:
        shutil.copy2(DATA / name, local / name)
    write_json(local / "generation_config.json", CONFIG["generation"])
    (local / "generate.py").write_text(INFERENCE_SOURCE)
    (local / "requirements-inference.txt").write_text(INFERENCE_REQUIREMENTS)
    assets = {
        str(p.relative_to(local)): sha(p)
        for p in local.rglob("*")
        if p.is_file() and p.name not in ["bundle_manifest.json", "manifest.json"]
    }
    write_json(
        local / "bundle_manifest.json",
        {
            "bundle_kind": "hw2_colab_inference",
            "question_count": 1919,
            "asset_sha256": assets,
            "identity": IDENTITY,
            "candidate": record,
            "weights_sha256": assets["model/model.safetensors"],
            "numerical_validation_accuracy": None,
            "public_pass_at_1": None,
            "selection": "provisional token-weighted validation CE",
            "own_weights_originally_random_initialization": True,
            "optimizer_or_train_included": False,
        },
    )
    write_json(
        local / "manifest.json",
        {
            "files": {
                name: {"sha256": digest, "bytes": (local / name).stat().st_size}
                for name, digest in assets.items()
            }
        },
    )
    remote = DRIVE_RUN / "exports" / record["name"]
    # Assets first, manifests last; partial trees are not advertised by completion.json.
    for p in local.rglob("*"):
        if p.is_file() and p.name not in ["bundle_manifest.json", "manifest.json"]:
            dest = remote / p.relative_to(local)
            if not dest.exists() or sha(dest) != sha(p):
                publish_file(p, dest)
    publish_file(local / "manifest.json", remote / "manifest.json")
    publish_file(local / "bundle_manifest.json", remote / "bundle_manifest.json")
    verify_tree(remote)
    loaded = BartForConditionalGeneration.from_pretrained(
        remote / "model", local_files_only=True, attn_implementation="sdpa"
    )
    if sum(x.numel() for x in loaded.parameters()) != PARAMETERS or any(
        not bool(torch.isfinite(x).all()) for x in loaded.parameters()
    ):
        raise ValueError("Saved Drive model invalid")
    del loaded
    return remote


def make_review_zip(status):
    out = RUN / "review_bundle.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in ["configs", "logs", "validation_predictions"]:
            for p in (RUN / folder).rglob("*"):
                if p.is_file():
                    z.write(p, str(p.relative_to(RUN)))
        for p in (DATA / "evaluation").glob("*"):
            if p.is_file():
                z.write(p, "evaluation/" + p.name)
        z.writestr(
            "review_metadata.json",
            json.dumps(
                {
                    "identity": IDENTITY,
                    "status": status,
                    "main_state": MAIN_STATE,
                    "numerical_validation_accuracy": None,
                    "public_pass_at_1": None,
                    "generated_programs_executed": False,
                },
                indent=2,
            ),
        )
    return out


if RUN_PHASES:
    VERIFIED_EXPORTS = []
    COMPLETION = None
    try:
        records = list(CANDIDATES)
        final_info = RUN / "exports/final/candidate.json"
        if final_info.exists():
            records.append(json.loads(final_info.read_text()))
        for record in records:
            BUDGET.require(60, final=True)
            local = RUN / "exports" / record["name"]
            remote = DRIVE_RUN / "exports" / record["name"]
            if not (local / "model/model.safetensors").exists():
                shutil.copytree(remote, local, dirs_exist_ok=True)
            saved = complete_portable_export(local, record)
            VERIFIED_EXPORTS.append(
                {
                    "name": record["name"],
                    "path": str(saved),
                    "CE": record["ce"],
                    "load_from_Drive_passed": True,
                }
            )
            if BUDGET.remaining(True) > 180:
                m = (
                    BartForConditionalGeneration.from_pretrained(
                        local / "model", local_files_only=True, attn_implementation="sdpa"
                    )
                    .to(DEVICE)
                    .eval()
                )
                save_generations(m, record["name"] + "_512", QUESTIONS, BUDGET)
                del m
                clean_cuda()
            else:
                log_event(
                    "generation_skipped", candidate=record["name"], reason="final deadline reserve"
                )
        if not VERIFIED_EXPORTS:
            raise RuntimeError(
                "No trained export available; inspect diagnostic review, disconnect A100"
            )
        provisional = min(
            [r for r in VERIFIED_EXPORTS if r["CE"] is not None], key=lambda r: r["CE"]
        )
        selected = {
            "export_name": provisional["name"],
            "export_path": provisional["path"],
            "selection": "provisional CE only; local493 accuracy and public unknown",
            "identity": IDENTITY,
        }
        write_json(RUN / "configs/selected_export.json", selected)
        sync_small()
        review = make_review_zip(
            {
                "abort_reason": ABORT_REASON,
                "verified_exports": VERIFIED_EXPORTS,
                "selected": selected,
            }
        )
        publish_file(review, DRIVE_RUN / "review_bundle.zip")
        if DRIVE_ERROR is not None:
            raise RuntimeError(
                "An earlier Drive failure occurred; verify full resume checkpoint before claiming completion"
            )
        COMPLETION = {
            "finished_utc": datetime.now(timezone.utc).isoformat(),
            "a100_elapsed_seconds": BUDGET.elapsed(),
            "known_A100_budget_seconds": A100_BUDGET_SECONDS,
            "verified_exports": VERIFIED_EXPORTS,
            "selected_export": selected,
            "review_bundle": str(DRIVE_RUN / "review_bundle.zip"),
            "numerical_validation_accuracy": None,
            "public_pass_at_1": None,
            "all_training_updates_finished": True,
            "runtime_automatically_disconnected": False,
            "identity": IDENTITY,
        }
        write_json(RUN / "completion.json", COMPLETION)
        publish_file(RUN / "completion.json", DRIVE_RUN / "completion.json")
        print(
            "DRIVE EXPORTS VERIFIED AND LOADABLE. A100 МОЖНО ОТКЛЮЧАТЬ ВРУЧНУЮ: Runtime → Disconnect and delete runtime.",
            flush=True,
        )
        print(json.dumps(COMPLETION, indent=2))
    except Exception as exc:
        print("FINALIZATION INCOMPLETE:", repr(exc), "Local results:", RUN, flush=True)
        write_json(
            RUN / "logs/export_error.json",
            {"error": repr(exc), "local_run": str(RUN), "verified_exports": VERIFIED_EXPORTS},
        )
        try:
            sync_small()
            publish_file(
                make_review_zip({"error": repr(exc), "verified_exports": VERIFIED_EXPORTS}),
                DRIVE_RUN / "review_bundle.zip",
            )
        except Exception:
            pass
        print(
            "Не считайте Drive-копию завершённой. Скачайте локальные результаты до ручного отключения A100. Python не отключает runtime."
        )


## После запуска

Скачайте `runs/<RUN_NAME>/review_bundle.zip` для локальной численной проверки493 references. Не отправляйте CE или review ZIP боту. На локальной машине команда описана в README; generated-программы исполняются только в существующем Docker sandbox.

Когда `completion.json` подтверждает проверку Drive, отключите A100 вручную, откройте самостоятельный `Colab_inference_T4.ipynb` в **новой T4-сессии**. По умолчанию он возьмёт последний завершённый run и предварительный CE-экспорт; после локальной оценки можно явно выбрать другой `EXPORT_PATH`. Перед подачей боту получите фактический T4 timing. Завершение кода не прекращает оплату runtime.